# Experiments Scraping Paper

In [1]:
import os
import sys
import json
from pathlib import Path
from dotenv import load_dotenv

In [2]:
# ─── 1. SETUP PATH KE BACKEND PACKAGE ───
# Menghitung path root (Tugas_Akhir) dari posisi notebook ini
ROOT_DIR = Path(os.getcwd()).parent.parent
# Menambahkan folder backend/package ke sys.path agar module 'knowledge' bisa di-import
BACKEND_PKG_PATH = ROOT_DIR / "backend" / "package"
if str(BACKEND_PKG_PATH) not in sys.path:
    sys.path.append(str(BACKEND_PKG_PATH))

In [3]:
# Load .env file untuk API Keys (SerpAPI, BrightData, dll)
env_file = ROOT_DIR / ".env"
if env_file.exists():
    load_dotenv(env_file, override=True)
    print(f"✅ Loaded environment from {env_file}")
else:
    print("⚠️  .env file not found, pastikan API Keys sudah ada di OS environment.")

✅ Loaded environment from C:\Users\rizky_11yf1be\Desktop\Tugas_Akhir\.env


In [4]:
# ─── 2. IMPORT ETL WORKER LOGIC ───
# Kita langsung pakai fungsi enrich_single_paper yang ada di core scraper
from knowledge.etl.scraping.keyword_scraper import enrich_single_paper

In [7]:
# ─── 3. TEST PARSING PAPER ───
# Contoh URL Google Scholar Citation (biasanya didapat dari kolom 'scholar_citation_url')
# Ganti URL ini dengan paper yang ingin kamu tes parsing-nya
test_url = "https://scholar.google.com/citations?view_op=view_citation&hl=en&user=hn5jrnAAAAAJ&pagesize=100&sortby=pubdate&citation_for_view=hn5jrnAAAAAJ:kRWSkSYxWN8C"
print(f"🚀 Menjalankan ETL Worker Enrichment untuk:\n{test_url}\n")

try:
    # Fungsi ini akan otomatis:
    # 1. Resolve citation lewat SerpAPI (Abstract, Title, Link)
    # 2. Scrape halaman Publisher via BrightData Proxy (Keywords, Full Abstract)
    # 3. Lookup CrossRef/OpenAlex jika DOI belum ada
    # 4. Fetch TLDR dari Semantic Scholar
    result = enrich_single_paper(test_url)
    print("\n" + "="*40)
    print("         HASIL SCRAPING")
    print("="*40)
    print(json.dumps(result, indent=2))
    
except Exception as e:
    print(f"❌ Terjadi kesalahan: {e}")

🚀 Menjalankan ETL Worker Enrichment untuk:
https://scholar.google.com/citations?view_op=view_citation&hl=en&user=hn5jrnAAAAAJ&pagesize=100&sortby=pubdate&citation_for_view=hn5jrnAAAAAJ:kRWSkSYxWN8C

   [SerpAPI] Resolving citation...
   [SerpAPI] Publisher: http://jeeemi.org/index.php/jeeemi/article/view/1215...
   [SerpAPI] Abstract: Adapting learning methods to individual learning styles rema...
   [Publisher] Scraping for keywords/DOI...
         Proxy Status: 407. Falling back to direct...
      [Publisher]   Keywords dari HTML: Keywords:, chatbot, whatsapp, learning styles, Expert System...
      [Publisher]   Doc Type dari meta tags: Electronics
   [Publisher] Keywords: keywords:, chatbot, whatsapp, learning styles, expert system...
   [Publisher] DOI: 10.35882/jeeemi.v8i1.1215
   [S2] TLDR: A rule-based adaptive WhatsApp chatbot capable of automatica...

         HASIL SCRAPING
{
  "abstract": "Adapting learning methods to individual learning styles remains a major challenge in 

In [8]:
from knowledge.etl.scraping.utils import clean_name_expert, normalize_name, fuzzy_match_name
from knowledge.etl.scraping.simcv_client import SimCVClient

In [9]:
# ─── 2. TEST MAPPING NAMA (SIMULASI) ───
# Kita tes apakah berbagai variasi nama akan merujuk ke orang yang sama
reference_name = "Rizky Yanuar"
test_cases = [
    "Prof. Dr. Rizky Yanuar, M.T.",
    "Ir. H. Rizky Yanuar, S.T., M.Kom.",
    "Rizky Yanuar (Dosen Informatika)",
    "Rizky Yanuar.",
    "Rizky Y."
]

print(f"🎯 Target Matching: '{reference_name}'\n")
print(f"{'Input Nama':<40} | {'Cleaned':<20} | {'Match Status'}")
print("-" * 85)

for name in test_cases:
    cleaned = clean_name_expert(name) # Menghapus gelar depan & belakang
    is_match, score, method = fuzzy_match_name(name, reference_name)
    
    status = f"✅ YES ({method}, score: {score:.2f})" if is_match else f"❌ NO ({score:.2f})"
    print(f"{name:<40} | {cleaned:<20} | {status}")


🎯 Target Matching: 'Rizky Yanuar'

Input Nama                               | Cleaned              | Match Status
-------------------------------------------------------------------------------------
Prof. Dr. Rizky Yanuar, M.T.             | Rizky Yanuar         | ✅ YES (exact, score: 1.00)
Ir. H. Rizky Yanuar, S.T., M.Kom.        | Rizky Yanuar         | ✅ YES (exact, score: 1.00)
Rizky Yanuar (Dosen Informatika)         | Rizky Yanuar (Dosen Informatika) | ✅ YES (contain, score: 1.00)
Rizky Yanuar.                            | Rizky Yanuar.        | ✅ YES (exact, score: 1.00)
Rizky Y.                                 | Rizky Y.             | ✅ YES (contain, score: 1.00)


In [11]:
# ─── 3. TEST LIVE SEARCH (SIMCV UNESA) ───
# Ini untuk memastikan pipeline bisa menemukan data dosen di database internal UNESA
print("\n" + "="*40)
print("🔎 TESTING LIVE SEARCH (SIMCV API)")
print("="*40)

client = SimCVClient()
search_query = "Riskyana Dewi Intan Puspitasari" # Masukkan nama dosen yang ingin dicari

print(f"Searching SimCV for: '{search_query}'...")
results = client.search(search_query)

if results:
    print(f"Found {len(results)} potential candidates:")
    for res in results:
        print(f"\n- Nama Lengkap: {res.get('namalengkap')}")
        print(f"  NIP/NIDN    : {res.get('nip')} / {res.get('nidn')}")
        print(f"  Prodi       : {res.get('namasatker')}")
        
        # Cek kecocokan fuzzy dengan query
        match, score, _ = fuzzy_match_name(search_query, res.get('namalengkap'))
        print(f"  Fuzzy Match : {'✅ Matched' if match else '⚠️ Low Score'} ({score:.2f})")
else:
    print("❌ No results found in SimCV. Coba cek typo atau gunakan nama tanpa gelar.")


🔎 TESTING LIVE SEARCH (SIMCV API)
Searching SimCV for: 'Riskyana Dewi Intan Puspitasari'...
Found 1 potential candidates:

- Nama Lengkap: Riskyana Dewi Intan Puspitasari, M.Kom.
  NIP/NIDN    : 199405212022032028 / 0021059403
  Prodi       : Kecerdasan Artifisial S1
  Fuzzy Match : ✅ Matched (1.00)
